In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

import random, math

seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)


In [16]:

# ==========================================================
# Cell 2: data loading & preprocessing
# ==========================================================
# If running on Colab with Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv("/content/drive/MyDrive/ECG_10s_2500.csv")

# Otherwise, set your local path:
df = pd.read_csv("/content/drive/MyDrive/ECG_10s_2500.csv")

print("Data shape (raw):", df.shape)
if 'label' not in df.columns:
    raise ValueError("Expected a 'label' column in the DataFrame.")
df = df.drop(columns=['patient_id'], errors='ignore')

# First 2500 columns are ECG samples
signal_cols = df.columns[:2500]
X = df[signal_cols].values.astype(np.float32)
y = df['label'].astype(int).values

# Per-segment z-normalization (row-wise), keep normalized scale for GAN and classifier
X_mean = X.mean(axis=1, keepdims=True)
X_std  = X.std(axis=1, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

timesteps = X.shape[1]  # 2500
X = X.reshape((-1, timesteps, 1))

print("X shape:", X.shape)               # (N, 2500, 1)
print("Label distribution:", np.bincount(y))


Data shape (raw): (45630, 2505)
X shape: (45630, 2500, 1)
Label distribution: [43169  2461]


In [17]:
# ==========================================================
# Cell 3: stratified split (70/15/15)
# ==========================================================
indices = np.arange(len(X))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=y, random_state=seed
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=seed+1
)

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print("Train:", X_train.shape, "Pos:", y_train.sum())
print("Val:  ", X_val.shape,   "Pos:", y_val.sum())
print("Test: ", X_test.shape,  "Pos:", y_test.sum())
print("Train pos ratio:", y_train.mean())


Train: (31941, 2500, 1) Pos: 1723
Val:   (6844, 2500, 1) Pos: 369
Test:  (6845, 2500, 1) Pos: 369
Train pos ratio: 0.05394320778936164


In [38]:
# ==========================================================
# Cell 4: Improved 1D GAN for synthetic positive samples
# ==========================================================

# Extract positive training samples for GAN
X_pos = X_train[y_train == 1]  # (n_pos, timesteps, 1)
n_pos = len(X_pos)
print("GAN training set (positives only):", X_pos.shape)

if n_pos == 0:
    print("No positive samples → GAN will be skipped.")
    # define dummies so later cells don't crash
    latent_dim = 100
    generator = tf.keras.Sequential(name="dummy_generator")
    discriminator = tf.keras.Sequential(name="dummy_discriminator")
else:
    # ------------------------------------------------------
    # Hyperparameters
    # ------------------------------------------------------
    timesteps = X_pos.shape[1]       # should be 2500
    latent_dim = 128                 # a bit larger latent space
    batch_size = 64
    steps = 6000                     # total training iterations
    print_interval = 200

    # ratio between D and G updates; keep D slightly ahead
    d_updates_per_step = 1
    g_updates_per_step = 1

    # ------------------------------------------------------
    # Improved 1D Generator
    # ------------------------------------------------------
    # ------------------------------------------------------
    # Improved 1D Generator (Sequential version to avoid Keras call issues)
    # ------------------------------------------------------
    def make_generator(latent_dim=128, out_len=2500):
        """
        1D conv generator:
        z -> Dense -> (625, 64) -> Upsample x2 -> (1250, 64)
           -> Upsample x2 -> (2500, 64) -> Conv1D -> (2500, 1)
        """
        base_len = out_len // 4  # 2500 // 4 = 625

        model = tf.keras.Sequential(name="generator_1d_ecg")
        model.add(layers.Input(shape=(latent_dim,)))
        model.add(layers.Dense(base_len * 64))
        model.add(layers.LeakyReLU(0.2))
        model.add(layers.Reshape((base_len, 64)))

        model.add(layers.UpSampling1D(size=2))
        model.add(layers.Conv1D(64, kernel_size=7, padding="same"))
        model.add(layers.LeakyReLU(0.2))
        model.add(layers.BatchNormalization())

        model.add(layers.UpSampling1D(size=2))
        model.add(layers.Conv1D(32, kernel_size=7, padding="same"))
        model.add(layers.LeakyReLU(0.2))
        model.add(layers.BatchNormalization())

        # final conv to single channel, linear activation
        model.add(layers.Conv1D(1, kernel_size=7, padding="same", activation="linear"))

        return model

    # ------------------------------------------------------
    # Improved 1D Discriminator
    # ------------------------------------------------------
    def make_discriminator(input_len=2500):
        """
        1D conv discriminator with dropout & LeakyReLU.
        """
        inp = layers.Input(shape=(input_len, 1))

        x = layers.Conv1D(32, kernel_size=7, strides=2, padding="same")(inp)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Conv1D(64, kernel_size=7, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Conv1D(128, kernel_size=7, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Flatten()(x)
        x = layers.Dense(128)(x)
        x = layers.LeakyReLU(0.2)(x)
        out = layers.Dense(1, activation="sigmoid")(x)

        model = tf.keras.Model(inp, out, name="discriminator_1d_ecg")
        return model

    generator = make_generator(latent_dim=latent_dim, out_len=timesteps)
    discriminator = make_discriminator(input_len=timesteps)

    # ------------------------------------------------------
    # Optimizers & compilation
    #   - Slightly higher LR for D than for G
    #   - GAN tricks: beta_1=0.5, beta_2=0.9
    # ------------------------------------------------------
    d_opt = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5, beta_2=0.9)
    g_opt = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)

    discriminator.compile(
        optimizer=d_opt,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    # combined model (for generator updates)
    z_in = layers.Input(shape=(latent_dim,))
    fake_out = generator(z_in)
    discriminator.trainable = False  # freeze D for G updates
    validity = discriminator(fake_out)
    combined = tf.keras.Model(z_in, validity, name="gan_combined")

    combined.compile(
        optimizer=g_opt,
        loss="binary_crossentropy",
    )

    # Re-enable trainability for D for when we call train_on_batch on it
    discriminator.trainable = True

    # ------------------------------------------------------
    # GAN training loop with:
    #   - label smoothing
    #   - label noise
    #   - instance noise on inputs
    # ------------------------------------------------------
    def smooth_labels(size, real=True):
        if real:
            # real labels in [0.8, 1.0]
            return np.random.uniform(0.8, 1.0, size=(size, 1)).astype(np.float32)
        else:
            # fake labels in [0.0, 0.2]
            return np.random.uniform(0.0, 0.2, size=(size, 1)).astype(np.float32)

    def add_instance_noise(x, sigma=0.05):
        noise = np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)
        return (x + noise).astype(np.float32)

    print("Starting GAN training...")
    for step in range(1, steps + 1):
        # -----------------------------
        # Train Discriminator
        # -----------------------------
        for _ in range(d_updates_per_step):
            idx = np.random.randint(0, X_pos.shape[0], size=batch_size)
            real_batch = X_pos[idx].astype(np.float32)

            # instance noise on real
            real_batch = add_instance_noise(real_batch, sigma=0.05)

            z = np.random.normal(0, 1, (batch_size, latent_dim)).astype(np.float32)
            fake_batch = generator.predict(z, verbose=0)

            # instance noise on fake
            fake_batch = add_instance_noise(fake_batch, sigma=0.05)

            y_real = smooth_labels(batch_size, real=True)
            y_fake = smooth_labels(batch_size, real=False)

            # occasionally flip a small fraction of labels (label noise)
            if np.random.rand() < 0.05:
                flip_idx = np.random.choice(batch_size, size=batch_size // 10, replace=False)
                y_real[flip_idx], y_fake[flip_idx] = y_fake[flip_idx], y_real[flip_idx]

            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(real_batch, y_real)
            d_loss_fake = discriminator.train_on_batch(fake_batch, y_fake)

            d_loss_real = np.array(d_loss_real)
            d_loss_fake = np.array(d_loss_fake)
            d_loss = 0.5 * (d_loss_real + d_loss_fake)

        # -----------------------------
        # Train Generator
        # -----------------------------
        for _ in range(g_updates_per_step):
            z = np.random.normal(0, 1, (batch_size, latent_dim)).astype(np.float32)
            # generator wants D to output "real" (smoothed)
            y_gen = smooth_labels(batch_size, real=True)

            discriminator.trainable = False
            g_loss = combined.train_on_batch(z, y_gen)

        # -----------------------------
        # Logging
        # -----------------------------
        if step % print_interval == 0 or step == 1:
            # d_loss can be scalar, list, or array; same for g_loss.
            d_arr = np.array(d_loss).ravel()
            d_loss_val = float(d_arr[0])
            d_acc_val = float(d_arr[1]) if d_arr.size > 1 else float("nan")

            g_arr = np.array(g_loss).ravel()
            g_loss_val = float(g_arr[0])

            print(
                f"[GAN step {step}/{steps}] "
                f"D_loss: {d_loss_val:.4f}, D_acc: {d_acc_val:.3f} | "
                f"G_loss: {g_loss_val:.4f}"
            )


    print("GAN training finished.")


GAN training set (positives only): (1723, 2500, 1)
Starting GAN training...


[GAN step 1/6000] D_loss: 0.7649, D_acc: 0.000 | G_loss: 0.3550
[GAN step 200/6000] D_loss: 0.3491, D_acc: 0.000 | G_loss: 0.3407
[GAN step 400/6000] D_loss: 0.3433, D_acc: 0.000 | G_loss: 0.3352
[GAN step 600/6000] D_loss: 0.3479, D_acc: 0.000 | G_loss: 0.5559
[GAN step 800/6000] D_loss: 0.3464, D_acc: 0.000 | G_loss: 0.8357
[GAN step 1000/6000] D_loss: 0.3554, D_acc: 0.000 | G_loss: 0.9296
[GAN step 1200/6000] D_loss: 0.3584, D_acc: 0.000 | G_loss: 1.0321
[GAN step 1400/6000] D_loss: 0.3605, D_acc: 0.000 | G_loss: 1.0994
[GAN step 1600/6000] D_loss: 0.3697, D_acc: 0.000 | G_loss: 1.0443
[GAN step 1800/6000] D_loss: 0.3910, D_acc: 0.000 | G_loss: 1.0394
[GAN step 2000/6000] D_loss: 0.4095, D_acc: 0.000 | G_loss: 1.0484
[GAN step 2200/6000] D_loss: 0.4191, D_acc: 0.000 | G_loss: 1.0552
[GAN step 2400/6000] D_loss: 0.4209, D_acc: 0.000 | G_loss: 1.0531
[GAN step 2600/6000] D_loss: 0.4228, D_acc: 0.000 | G_loss: 1.0892
[GAN step 2800/6000] D_loss: 0.4211, D_acc: 0.000 | G_loss: 1.0981
[G

In [39]:
# sample a batch of reals and fakes
idx = np.random.randint(0, X_pos.shape[0], size=512)
real_batch = X_pos[idx]
z = np.random.normal(0, 1, (512, latent_dim)).astype(np.float32)
fake_batch = generator.predict(z, verbose=0)

real_scores = discriminator.predict(real_batch, verbose=0).ravel()
fake_scores = discriminator.predict(fake_batch, verbose=0).ravel()

print("Real score mean±sd:", real_scores.mean(), real_scores.std())
print("Fake score mean±sd:", fake_scores.mean(), fake_scores.std())


Real score mean±sd: 0.9083661 0.0407335
Fake score mean±sd: 0.111895084 0.05879774


In [41]:
# ==========================================================
# Cell 5: Use GAN to synthesize extra positives to ~30% target ratio (FIXED)
# ==========================================================
target_pos_ratio = 0.30

pos_mask = (y_train == 1)
neg_mask = (y_train == 0)
n_pos = int(pos_mask.sum())
n_neg = int(neg_mask.sum())

desired_pos = (target_pos_ratio / (1.0 - target_pos_ratio)) * n_neg  # 0.3/0.7 * n_neg
m_needed = int(max(0, math.floor(desired_pos - n_pos + 1e-9)))

# Safety cap to avoid flooding with low-diversity fakes
max_synth = max(0, 2 * n_pos)                 # at most 2x real positives
m_needed = min(m_needed, max_synth)

print(f"Current train pos: {n_pos}, neg: {n_neg} → synth positives needed: {m_needed}")

def _per_segment_renorm(x):
    # normalize per-segment to match distribution of real ECG windows
    m = x.mean(axis=0, keepdims=True)
    s = x.std(axis=0, keepdims=True) + 1e-8
    return (x - m) / s

if m_needed > 0 and n_pos > 0:
    generated = 0
    kept_chunks = []

    while generated < m_needed:
        bs = min(batch_size, m_needed - generated)

        # Generate latent noise
        z = np.random.normal(0, 1, (bs, latent_dim)).astype(np.float32)

        # Generate fake ECG segments
        fake_batch = generator.predict(z, verbose=0).astype(np.float32)    # (bs, 2500, 1)

        # Optional: discriminator-based gating
        try:
            scores = discriminator.predict(fake_batch, verbose=0).reshape(-1)  # probability "real"
            keep_mask = scores > 0.5
            fake_batch = fake_batch[keep_mask]
        except Exception:
            # If D not ready, skip gating
            pass

        # If gating removed everything, fallback to the ungated batch
        if fake_batch.size == 0:
            fake_batch = generator.predict(z, verbose=0).astype(np.float32)

        # -------- FIXED RESHAPE BLOCK --------
        # Use true batch size after gating
        n_batch = fake_batch.shape[0]
        seq_len = fake_batch.shape[1]

        # Renormalize
        fake_batch = _per_segment_renorm(fake_batch.reshape(n_batch, -1))
        fake_batch = fake_batch.reshape(n_batch, seq_len, 1)

        # Clip outliers for safety
        fake_batch = np.clip(fake_batch, -3.5, 3.5).astype(np.float32)
        # --------------------------------------

        # Only add up to the amount still needed
        kept = min(fake_batch.shape[0], m_needed - generated)
        kept_chunks.append(fake_batch[:kept])
        generated += kept

    # Final concatenation of all synthetic positives
    X_synth = np.concatenate(kept_chunks, axis=0)[:m_needed]     # (m_needed, 2500, 1)
    y_synth = np.ones((X_synth.shape[0],), dtype=int)

    # Augment training set
    X_train_aug = np.concatenate([X_train, X_synth], axis=0).astype(np.float32)
    y_train_aug = np.concatenate([y_train, y_synth], axis=0)
else:
    print("No synthetic positives added (either none needed or no positives available to train GAN).")
    X_train_aug = X_train.copy().astype(np.float32)
    y_train_aug = y_train.copy()

# Shuffle augmented training set
perm = np.random.permutation(X_train_aug.shape[0])
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

pos_ratio = float(y_train_aug.mean())
print("After GAN aug - train:", X_train_aug.shape, "pos ratio:", f"{pos_ratio:.3f}")


Current train pos: 1723, neg: 30218 → synth positives needed: 3446
After GAN aug - train: (35387, 2500, 1) pos ratio: 0.146


In [42]:



# ==========================================================
# Cell 6: Classifier — CNN + BiLSTM (your chosen architecture)
# ==========================================================
def build_cnn_bilstm(timesteps, channels=1, lr=1e-3):
    inp = layers.Input(shape=(timesteps, channels))

    x = layers.Conv1D(32, kernel_size=7, strides=2, padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, kernel_size=7, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=7, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
        ]
    )
    return model

clf = build_cnn_bilstm(timesteps, channels=1)
clf.summary()

# ==========================================================
# Cell 7: Train classifier (epochs=100, patience=20)
# ==========================================================
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=20, restore_best_weights=True
)

history = clf.fit(
    X_train_aug, y_train_aug,
    epochs=100,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=2
)



Model: "functional_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_18 (InputLayer)     │ (None, 2500, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_40 (Conv1D)              │ (None, 1250, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 1250, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_6 (ReLU)                  │ (None, 1250, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 625, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_41 (Conv1D)              │ (None, 625, 64)        │        14,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 625, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_7 (ReLU)                  │ (None, 625, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 312, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_42 (Conv1D)              │ (None, 312, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 312, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_8 (ReLU)                  │ (None, 312, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 156, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 156, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,161 (703.75 KB)

 Trainable params: 179,713 (702.00 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/100
277/277 - 19s - 69ms/step - auc: 0.9052 - loss: 0.2201 - precision: 0.8647 - recall: 0.5945 - val_auc: 0.7624 - val_loss: 0.2372 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
277/277 - 12s - 44ms/step - auc: 0.9422 - loss: 0.1562 - precision: 0.9529 - recall: 0.7121 - val_auc: 0.8400 - val_loss: 0.2264 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
277/277 - 12s - 43ms/step - auc: 0.9515 - loss: 0.1419 - precision: 0.9577 - recall: 0.7353 - val_auc: 0.8876 - val_loss: 0.1555 - val_precision: 0.7333 - val_recall: 0.0596
Epoch 4/100
277/277 - 21s - 74ms/step - auc: 0.9605 - loss: 0.1290 - precision: 0.9598 - recall: 0.7578 - val_auc: 0.8984 - val_loss: 0.1579 - val_precision: 1.0000 - val_recall: 0.0244
Epoch 5/100
277/277 - 12s - 44ms/step - auc: 0.9714 - loss: 0.1166 - precision: 0.9559 - recall: 0.7717 - val_auc: 0.8769 - val_loss: 0.1519 - val_precision: 0.7581 - val_recall: 0.1274
Epoch 6/100
277/277 - 12s - 44ms/step - auc: 0.9766 - 

In [43]:

# ==========================================================
# Cell 8: Threshold tuning on validation
# ==========================================================
val_probs = clf.predict(X_val).ravel()
best_th = 0.5
best_f1 = -1.0
best_prec = 0.0
best_rec  = 0.0

for th in np.linspace(0.05, 0.95, 19):
    y_pred_val = (val_probs >= th).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_val, y_pred_val, average='binary', zero_division=0
    )
    if f1 > best_f1:
        best_f1, best_th, best_prec, best_rec = f1, th, prec, rec

print(f"Best threshold on val: {best_th:.3f}")
print(f"Val F1: {best_f1:.3f}, Recall: {best_rec:.3f}, Precision: {best_prec:.3f}")


214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Best threshold on val: 0.500
Val F1: 0.722, Recall: 0.667, Precision: 0.788


In [73]:

# ==========================================================
# Cell 9: Final test evaluation (uses tuned threshold)
# ==========================================================
test_probs = clf.predict(X_test).ravel()
y_pred_test = (test_probs >= 0.4).astype(int)

prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(
    y_test, y_pred_test, average='binary', zero_division=0
)
acc_t = (y_pred_test == y_test).mean()
auc_t = roc_auc_score(y_test, test_probs)
cm_t  = confusion_matrix(y_test, y_pred_test)

print("=== Test metrics (random split, NO patient-wise separation) ===")
print(f"Accuracy: {acc_t:.3f}")
print(f"Recall (hypo): {rec_t:.3f}")
print(f"Precision (hypo): {prec_t:.3f}")
print(f"F1 (hypo): {f1_t:.3f}")
print(f"AUROC: {auc_t:.3f}")
print("\nConfusion matrix (rows=true, cols=pred):\n", cm_t)


214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
=== Test metrics (random split, NO patient-wise separation) ===
Accuracy: 0.975
Recall (hypo): 0.778
Precision (hypo): 0.769
F1 (hypo): 0.774
AUROC: 0.976

Confusion matrix (rows=true, cols=pred):
 [[6390   86]
 [  82  287]]
